El objetivo es ejecutar el codigo realizado el día miércoles. En la lista de texto, encuentra 5 oraciones que den p_pos = 0.5 y p_neg = 0.5

In [35]:
%pip install transformers datasets scikit-learn torch

Note: you may need to restart the kernel to use updated packages.


In [36]:
from transformers import pipeline
import torch 

clf = pipeline(
    'text-classification',
    model='pysentimiento/robertuito-sentiment-analysis',
    tokenizer='pysentimiento/robertuito-sentiment-analysis',
    top_k=None,
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [37]:
from datasets import load_dataset
from collections import Counter 
import random 

raw = load_dataset("sepidmnorozy/Spanish_sentiment", split='train')
print(f"Numero de muestras: {len(raw)}. {raw.column_names}, {Counter(raw['label'])}")

Numero de muestras: 1029. ['label', 'text'], Counter({1: 851, 0: 178})


In [38]:
idx = list(range(len(raw)))
random.seed(42)
random.shuffle(idx)

n_val = int(0.2 * len(idx))
val_idx, train_idx = idx[:n_val], idx[n_val:]

train_texts = [raw[i]['text'] for i in train_idx]
train_labels = [int(raw[i]['label']) for i in train_idx]

val_texts = [raw[i]['text'] for i in val_idx]
val_labels = [int(raw[i]['label']) for i in val_idx]

In [39]:
for texto, label in zip(train_texts[:3], train_labels[:3]):
    print(label, texto)

1 Hermoso lugar !
1 Muy correcto en todos los aspectos , muy limpio , tiene detalles que agradan , como el mini · bar gratuito ( bebidas no alcoholicas ) y el calienta liquidos de la habitacion para que te puedas hacer cafe o infusiones.Reponen a diario .
1 En general muy bueno y el barrio donde esta situado es un barrio amable sencillo y de buenas costumbres .


In [40]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_name = "dccuchile/bert-base-spanish-wwm-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label={0: 'Negativo', 1:'Positivo'},
    label2id={'Negativo':0, 'Positivo':1}
).to(device)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING

In [41]:
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

class ReviewDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels 

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[i], dtype=torch.long)
        return item

def tokenize_texts(tokenizer, texts, max_length=128):
    return tokenizer(
        texts, 
        truncation=True,
        padding=True,
        max_length=max_length,
        return_tensors=None
    )

max_length = 128 
batch_size = 16

train_ds = ReviewDataset(
    tokenize_texts(tokenizer, train_texts, max_length),
    train_labels
)

val_ds = ReviewDataset(
    tokenize_texts(tokenizer, val_texts, max_length),
    val_labels
)

In [42]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup, set_seed

set_seed(42) # resultados reproducibles

optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
n_epochs = 3 # con 15 épocas el modelo se sobreajusta y da probabilidades de 0 o 1

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=batch_size)

total_steps = len(train_loader) * n_epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1*total_steps),
    num_training_steps=total_steps
)

for epoch in range(n_epochs):
    model.train()
    for batch in train_loader:
        labels = batch.pop('labels').to(device)
        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()
        loss = model(**batch, labels=labels).loss # extraer la perdida
        loss.backward() 

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1)
        optimizer.step()
        scheduler.step()


In [43]:
from sklearn.metrics import accuracy_score, f1_score

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    y_true, y_pred = [], []
    for batch in loader:
        labels = batch.pop('labels')
        batch = {k: v.to(device) for k, v in batch.items()}
        preds = model(**batch).logits.argmax(dim=-1).cpu().tolist()
        y_pred.extend(preds)
        y_true.extend(labels.tolist())

    return (
        accuracy_score(y_true, y_pred),
        f1_score(y_true, y_pred, average='macro'),
        y_true,
        y_pred
    )

In [44]:
from sklearn.metrics import classification_report, confusion_matrix

acc, f1, y_true, y_pred = evaluate(model, val_loader, device)

print(f"Accuracy: {acc*100:.2f}%")
print(f"F1-Score: {f1*100:.2f}%")

print(classification_report(
    y_true, y_pred,
    target_names=['Negativo', 'Positivo'],
    digits=2
))

Accuracy: 94.15%
F1-Score: 88.29%
              precision    recall  f1-score   support

    Negativo       0.89      0.73      0.80        33
    Positivo       0.95      0.98      0.97       172

    accuracy                           0.94       205
   macro avg       0.92      0.85      0.88       205
weighted avg       0.94      0.94      0.94       205



In [45]:
@torch.no_grad()
def predecir(textos, tokenizer, model, device, max_length=128):
    model.eval()
    enc = tokenizer(textos, truncation=True, padding=True,
                    max_length=max_length, return_tensors='pt')

    enc = {k: v.to(device) for k, v in enc.items()}
    probs = torch.softmax(model(**enc).logits, dim=-1).cpu() # score que tan seguro esta
    ids = probs.argmax(dim=-1).tolist() # 0 o 1

    return [
        {'texto': t, "label": {0: 'Negativo', 1: 'Positivo'}[i],
         'p_neg': float(p[0]), 'p_pos': float(p[1])}
         for t, i, p in zip(textos, ids, probs)
    ]

In [46]:
# Generar candidatas combinando una opinión positiva con una negativa.
# Las oraciones mixtas son las que dejan al modelo indeciso (p_pos ≈ p_neg ≈ 0.5).
import itertools

positivas = [
    "la habitación era cómoda", "el personal fue amable", "la ubicación es excelente",
    "el desayuno estaba rico", "la vista era bonita", "la cama era cómoda",
    "el precio era razonable", "la piscina estaba limpia", "el baño era amplio",
    "el wifi funcionaba bien",
]
negativas = [
    "el servicio fue lento", "había mucho ruido", "la limpieza dejaba que desear",
    "el aire acondicionado no funcionaba", "la comida era mala", "el personal fue grosero",
    "la habitación era pequeña", "el baño estaba sucio", "el precio era excesivo",
    "el wifi no funcionaba",
]

textos = []
for p, n in itertools.product(positivas, negativas):
    textos += [
        f"{p.capitalize()}, pero {n}.",
        f"{n.capitalize()}, pero {p}.",
        f"{p.capitalize()}, aunque {n}.",
        f"{n.capitalize()}, aunque {p}.",
    ]

print(f"{len(textos)} oraciones candidatas")

400 oraciones candidatas


In [47]:
resultados = predecir(textos, tokenizer, model, device)

In [48]:
# Ordenar por cercanía a 0.5 / 0.5 y mostrar las 5 mejores
resultados = sorted(resultados, key=lambda r: abs(r['p_pos'] - 0.5))

print("=== 5 oraciones más cercanas a p_pos = p_neg = 0.5 ===")
for r in resultados[:5]:
    print(f"[{r['label'].upper()}] P_NEG: {r['p_neg']:.4f} P_POS: {r['p_pos']:.4f}")
    print(f"    {r['texto']}")

=== 5 oraciones más cercanas a p_pos = p_neg = 0.5 ===
[POSITIVO] P_NEG: 0.4959 P_POS: 0.5041
    La limpieza dejaba que desear, aunque el wifi funcionaba bien.
[NEGATIVO] P_NEG: 0.5110 P_POS: 0.4890
    El wifi funcionaba bien, aunque el personal fue grosero.
[POSITIVO] P_NEG: 0.4889 P_POS: 0.5111
    Había mucho ruido, aunque el baño era amplio.
[NEGATIVO] P_NEG: 0.5131 P_POS: 0.4869
    El wifi no funcionaba, pero la piscina estaba limpia.
[NEGATIVO] P_NEG: 0.5200 P_POS: 0.4800
    La habitación era cómoda, pero el personal fue grosero.
